# 00 · Environment and data

**Question:** Which data and environment produced the recorded experiment?

This notebook renders recorded **competition-data** evidence. It verifies the committed artifact hashes, not private out-of-fold predictions. No credentials, raw comments, weight downloads, or training are needed. Full private metric recomputation remains `uv run jigsaw review`.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition data | 2,029 training rows | recorded local cross-validation")
print("Verification: aggregate file checksums and provenance; no model fitting.")

names = {"comment_only": "Comment-only TF-IDF", "rule_examples": "Rule/example TF-IDF",
         "semantic_margin": "Frozen semantic margin", "semantic_classifier": "Semantic classifier"}
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records):
    return pd.DataFrame([{"Model": names[r["model"]], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]} for r in records]).round(4)


Competition data | 2,029 training rows | recorded local cross-validation
Verification: aggregate file checksums and provenance; no model fitting.


## Reproduction environment
The original experiment environment and this notebook-execution environment are reported separately. `uv.lock` pins the project environment; rendering a historical result does not rerun that experiment.

In [2]:
recorded = baseline["provenance"]["environment"]
current = environment()
print("Recorded experiment Python:", recorded["python"])
print("Notebook execution Python:", current["python"])
display(pd.DataFrame({"Recorded experiment": recorded["packages"],
                      "Notebook execution": current["packages"]}))

Recorded experiment Python: 3.12.14
Notebook execution Python: 3.12.14


,Recorded experiment,Notebook execution
numpy,2.3.5,2.3.5
pandas,2.2.3,2.2.3
scipy,1.17.0,1.17.0
scikit-learn,1.8.0,1.8.0
joblib,1.5.3,1.5.3
plotly,7.0.0,7.0.0


## Recorded data contract
One row pairs a comment with a rule and examples. These are the saved audit counts, not a fresh read of raw CSV files. The preview test is not an independent evaluation set.

In [3]:
audit = baseline["audit"]
display(pd.DataFrame({"Recorded file": ["train.csv", "test.csv (preview)"],
                      "Rows": [audit["train_rows"], audit["preview_test_rows"]]}))
print("Training SHA-256:", baseline["training_sha256"])
print("Lexical run:", baseline["run_id"])
print("Semantic run:", semantic["run_id"])

,Recorded file,Rows
0,train.csv,2029
1,test.csv (preview),10


Training SHA-256: 83948d06a1e4b16421b738add60ef489cf1d44a2349ca711958fbb41c6207a0a
Lexical run: c15c2c2318fc0ed619c6
Semantic run: 4e7e6c00d269c451c0a3


## Continue
[01 · Data and validation](01_data_and_validation.ipynb) explains the leakage controls. For the employer overview, start with [03 · Results](03_saved_results.ipynb). To audit raw files locally, use `uv run jigsaw audit`; raw data remain private.